# Syllabus
- Basic Prompting
- Few-shot Learning
- Structured Prompts
- Structured Output

In [10]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

load_dotenv()

True

## Basic Prompting

In [51]:
model = init_chat_model(model="gpt-5.4-nano", temperature=0.7)

# Product complaint text
complaint_text = """
I ordered a smartwatch from your website last week, and it arrived with a cracked screen. The packaging was flimsy, and there was no protective material. I called customer service, but they told me they couldn’t help me without a return receipt, which I didn’t get. This is very frustrating. I need a replacement or refund immediately!
"""

In [52]:
# Initialize the language model (e.g., GPT-3 or GPT-4)
agent = create_agent(model=model)
response = agent.invoke({"messages": [HumanMessage(content=complaint_text)]})
print(response['messages'][-1].content)

I’m really sorry this happened—an item arriving with a cracked screen is not okay, especially if the packaging didn’t protect it. I can help, but I need to clarify one thing: I don’t have access to your order history or the ability to process refunds/replacements directly from here.

That said, you can usually get this resolved immediately by using the right evidence and the right channel. Here’s what to do:

### 1) Gather proof (do this now)
- Photos/videos of:  
  - the cracked screen (close + wide shots)  
  - the packaging (especially showing it was flimsy and lacked protective material)  
  - any shipping label and any damage to the box  
- Photos of the order confirmation (email/order page screenshot)
- Any tracking number or delivery confirmation you have

### 2) Contact customer support again—request **DOA/arrived-damaged** replacement/refund
When you contact them, use “arrived-damaged / defective on arrival” rather than “return,” because many retailers don’t require a return r

In [53]:
# Define system prompt for customer support agent
system_prompt = """
You are a customer support agent with excellent empathy. Your task is to resolve complaints by acknowledging the issue, apologizing, and providing clear next steps for the customer. Your response should focus on the following:
1. Empathizing with the customer.
2. Acknowledging the specific complaint (e.g., product damage, delivery issues).
3. Offering a concrete resolution (e.g., replacement, refund, or next steps).
Be professional, kind, and concise.
"""

agent = create_agent(model=model, system_prompt=system_prompt)

response = agent.invoke({"messages": [HumanMessage(content=review_text)]})
print(response['messages'][-1].content)


Thanks for taking the time to share your experience—I'm really glad to hear the blender is working well for you and that the simple controls and performance are meeting your expectations.  

On the one point you mentioned: it *can* get louder with tougher ingredients, which is common for blenders under heavier load. If you’d like, I can help you reduce the noise and improve results (e.g., tips on ingredient size, processing time, and load levels).  

Quick question so I can guide you best:  
- What kinds of “tougher ingredients” are you blending when it gets loud (ice, frozen fruit, nuts, etc.)?  

If you’re noticing anything unusual—like a grinding sound, burning smell, or a sudden increase in noise—tell me and we can check whether it might need service or an exchange.


## Few-shot Learning

In [54]:
user_message = "I want to wear polo shirt with shoes."

# Few-shot prompt with examples
system_prompt = """
You are a stylist. Help the user to find the perfect color for their clothes.

user: I want to wear boots with jacket.
stylist: boot=black, jacket=burgundy

user: I  want to wear jeans with shirts for my date.
stylist: jeans=light blue, shirts=stripped black and red
"""

agent = create_agent(model=model, system_prompt=system_prompt)

response = agent.invoke({"messages": [HumanMessage(content=user_message)]})
print(response['messages'][-1].content)


Sure! For a clean match:

- **Polo shirt:** **white**
- **Shoes:** **black** (or **brown** if you want a warmer look)

If you tell me the **color of your polo (or your skin tone / vibe: sporty vs classic)**, I can fine-tune it.


## Structured Prompts

In [55]:
user_message = "I want to wear polo shirt with shoes."

# Few-shot prompt with examples
system_prompt = """
You are a stylist. Help the user to find the perfect color for their clothes.
Please keep to the below structure.


name: name of the style

then for each garment:
    garment: name of the garment
    color: color name
"""

agent = create_agent(model=model, system_prompt=system_prompt)

response = agent.invoke({"messages": [HumanMessage(content=user_message)]})
print(response['messages'][-1].content)


name: Clean Sporty Match

garment: polo shirt  
color: Navy

garment: shoes  
color: White sneakers


## Structured Outputs

In [65]:
from pydantic import BaseModel, Field

user_message = "I want to wear a hoodie with nike hats."

# Few-shot prompt with examples
system_prompt = """You are a stylist. Help the user to find the perfect color for each piece of their garment."""

class StyleFormat(BaseModel):
    name: str = Field(description="a cool name for the style. it's like a title for this style. for example: 'hot night'.")
    colors: dict[str, str] = Field(description="a mapping between every single garment and its color. for example {'jeans': 'dark blue'}. Don't include any garment that is not included in user input. Only suggest one color per garment.")

agent = create_agent(
    model='gpt-5.4-nano',
    system_prompt=system_prompt,
    response_format=StyleFormat
)

question = HumanMessage(content=user_message)

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

StyleFormat(name='Streetwear Match-Up', colors={'hoodie': 'midnight navy', 'nike hats': 'classic black'})